In [5]:
import datetime as dt
import requests
import pandas as pd

# --- настройки ---
SECID = "SBER"  # поменяйте на нужный тикер, например: GAZP, LKOH, YNDX, TCSG

# Последние ~30 дней (календарные). Если нужны строго торговые дни — это тоже можно сделать.
till = dt.date.today()
from_ = till - dt.timedelta(days=30)

url = f"https://iss.moex.com/iss/engines/stock/markets/shares/securities/{SECID}/candles.json"
params = {
    "from": from_.isoformat(),
    "till": till.isoformat(),
    "interval": 24,   # 24 = дневные свечи
    "iss.meta": "off",
    "iss.only": "candles",
}

rows = []
start = 0
while True:
    r = requests.get(url, params={**params, "start": start}, timeout=30)
    r.raise_for_status()
    payload = r.json()["candles"]
    cols = payload["columns"]
    data = payload["data"]

    if not data:
        break

    rows.extend(data)
    start += len(data)

candles = pd.DataFrame(rows, columns=cols)
if candles.empty:
    raise RuntimeError(f"Нет данных по {SECID} за период {from_}..{till}. Проверьте SECID/рынок.")

# Средняя цена за месяц: среднее арифметическое дневных цен закрытия
candles["begin"] = pd.to_datetime(candles["begin"])
avg_close = float(pd.to_numeric(candles["close"], errors="coerce").dropna().mean())

print(f"{SECID}: средняя цена закрытия за последние 30 дней ({from_}..{till}) = {avg_close:.2f} RUB")
print(f"Дней в выборке: {len(candles)}")

candles[["begin", "open", "high", "low", "close", "volume"]].sort_values("begin").tail(10)

SBER: средняя цена закрытия за последние 30 дней (2026-04-03..2026-05-03) = 320.77 RUB
Дней в выборке: 30


,begin,open,high,low,close,volume
20,2026-04-23,326.60,328.24,325.80,327.39,20016804
21,2026-04-24,327.69,328.17,324.23,325.19,29137930
22,2026-04-25,325.23,325.73,325.23,325.68,1021141
23,2026-04-26,325.71,325.95,324.81,324.98,1288257
24,2026-04-27,325.28,325.69,321.61,321.99,17284762
25,2026-04-28,322.04,322.89,319.40,319.82,24479673
26,2026-04-29,320.20,321.85,316.21,320.09,40766275
27,2026-04-30,321.99,321.99,318.21,319.81,18920593
28,2026-05-02,320.35,320.67,320.04,320.58,626949
29,2026-05-03,320.65,321.05,320.31,320.96,615957


In [6]:
import numpy as np

# --- Датасет: событие за ~21 торговый день (условный «месяц») ---
# y_hit_within = 1, если среди close следующих HORIZON_TRADING_DAYS торговых дней есть значение
# строго выше сегодняшнего close (макс. будущих закрытий > текущее закрытие).

TICKERS = [
    "SBER", "GAZP", "LKOH", "GMKN", "ROSN",
    "NVTK", "TATN", "MGNT", "ALRS", "VTBR",
]

HORIZON_TRADING_DAYS = 21

def fetch_candles(secid: str, from_date: dt.date, till_date: dt.date, interval: int = 24) -> pd.DataFrame:
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/securities/{secid}/candles.json"
    params = {
        "from": from_date.isoformat(),
        "till": till_date.isoformat(),
        "interval": interval,
        "iss.meta": "off",
        "iss.only": "candles",
    }

    rows = []
    start = 0
    while True:
        r = requests.get(url, params={**params, "start": start}, timeout=30)
        r.raise_for_status()
        payload = r.json()["candles"]
        cols = payload["columns"]
        data = payload["data"]
        if not data:
            break
        rows.extend(data)
        start += len(data)

    df = pd.DataFrame(rows, columns=cols)
    if df.empty:
        return df

    df = df.copy()
    df["secid"] = secid
    df["begin"] = pd.to_datetime(df["begin"])
    for c in ["open", "high", "low", "close", "value", "volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def make_features(df: pd.DataFrame, horizon: int = HORIZON_TRADING_DAYS) -> pd.DataFrame:
    df = df.sort_values(["secid", "begin"]).copy()

    # базовые доходности
    df["ret1"] = df.groupby("secid")["close"].pct_change(1)
    df["ret2"] = df.groupby("secid")["close"].pct_change(2)
    df["ret5"] = df.groupby("secid")["close"].pct_change(5)

    # внутридневные признаки
    df["hl_spread"] = (df["high"] - df["low"]) / df["close"]
    df["co"] = (df["close"] - df["open"]) / df["open"]

    # объём
    df["vol_chg1"] = df.groupby("secid")["volume"].pct_change(1)

    # скользящие средние и волатильность
    g = df.groupby("secid")
    df["ma5"] = g["close"].transform(lambda s: s.rolling(5).mean())
    df["ma20"] = g["close"].transform(lambda s: s.rolling(20).mean())
    df["ma_ratio"] = df["ma5"] / df["ma20"] - 1.0

    df["vol5"] = g["ret1"].transform(lambda s: s.rolling(5).std())
    df["vol20"] = g["ret1"].transform(lambda s: s.rolling(20).std())
    df["vol_ratio"] = df["vol5"] / df["vol20"]

    # таргет: в ближайшие `horizon` торговых дней max(close) > сегодняшнее close
    shift_list = [
        g["close"].transform(lambda s, kk=kk: s.shift(-kk))
        for kk in range(1, horizon + 1)
    ]
    df["fwd_max_close"] = pd.concat(shift_list, axis=1).max(axis=1)
    df["y_hit_within"] = (df["fwd_max_close"] > df["close"]).astype("int")

    feats = ["ret1", "ret2", "ret5", "hl_spread", "co", "vol_chg1", "ma_ratio", "vol_ratio"]
    out = df[["secid", "begin", "close", "y_hit_within"] + feats].copy()
    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    return out

# период обучения
train_till = dt.date.today()
train_from = train_till - dt.timedelta(days=365 * 3)  # ~3 года

all_candles = []
for t in TICKERS:
    c = fetch_candles(t, train_from, train_till, interval=24)
    if not c.empty:
        all_candles.append(c)

candles_all = pd.concat(all_candles, ignore_index=True) if all_candles else pd.DataFrame()
if candles_all.empty:
    raise RuntimeError("Не удалось собрать свечи. Проверьте тикеры/доступность ISS.")

ds = make_features(candles_all)
t_dum = pd.get_dummies(ds["secid"], prefix="t", dtype=int)
ds = pd.concat([ds.reset_index(drop=True), t_dum.reset_index(drop=True)], axis=1)

print("Размер датасета:", ds.shape)
print(f"Горизонт: {HORIZON_TRADING_DAYS} торговых дней вперёд")
print("Доля y_hit_within=1:", ds["y_hit_within"].mean().round(3))

ds.sort_values(["secid", "begin"]).tail(5)

Размер датасета: (8316, 22)
Горизонт: 21 торговых дней вперёд
Доля y_hit_within=1: 0.85


,secid,begin,close,y_hit_within,ret1,ret2,ret5,hl_spread,co,vol_chg1,...,t_ALRS,t_GAZP,t_GMKN,t_LKOH,t_MGNT,t_NVTK,t_ROSN,t_SBER,t_TATN,t_VTBR
8311,VTBR,2026-04-28,92.640,1,-0.007180,-0.015881,-0.014678,0.016839,-0.007287,0.329290,...,0,0,0,0,0,0,0,0,0,1
8312,VTBR,2026-04-29,92.860,0,0.002375,-0.004823,-0.014643,0.066875,0.002375,3.630279,...,0,0,0,0,0,0,0,0,0,1
8313,VTBR,2026-04-30,91.300,0,-0.016799,-0.014465,-0.030631,0.027875,-0.017540,-0.765463,...,0,0,0,0,0,0,0,0,0,1
8314,VTBR,2026-05-02,91.295,0,-0.000055,-0.016853,-0.030169,0.003177,-0.000055,-0.973583,...,0,0,0,0,0,0,0,0,0,1
8315,VTBR,2026-05-03,91.030,0,-0.002903,-0.002957,-0.024435,0.004998,-0.002903,1.608096,...,0,0,0,0,0,0,0,0,0,1


In [7]:
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    brier_score_loss,
)

# Сплит по уникальным датам: все тикеры за день целиком в train или test (без «реза» дня пополам).
ds_sorted = ds.sort_values(["begin", "secid"]).reset_index(drop=True)
day = pd.to_datetime(ds_sorted["begin"]).dt.normalize()
unique_dates = sorted(day.unique())
n_dates = len(unique_dates)
cut_i = max(1, int(n_dates * 0.8))
train_dates = set(unique_dates[:cut_i])
test_dates = set(unique_dates[cut_i:])
if len(test_dates) == 0:
    raise RuntimeError("Недостаточно дат для теста — увеличьте период выгрузки.")

train = ds_sorted.loc[day.isin(train_dates)].copy()
test = ds_sorted.loc[day.isin(test_dates)].copy()

feature_cols = [
    c for c in ds_sorted.columns
    if c not in ("secid", "begin", "close", "y_hit_within")
    and pd.api.types.is_numeric_dtype(ds_sorted[c])
]
X_train, y_train = train[feature_cols], train["y_hit_within"]
X_test, y_test = test[feature_cols], test["y_hit_within"]

base_clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# Изотоническая калибровка вероятностей на временных фолдах обучающей выборки
cal_clf = CalibratedClassifierCV(
    base_clf,
    cv=TimeSeriesSplit(n_splits=3),
    method="isotonic",
)
cal_clf.fit(X_train, y_train)

proba = cal_clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)
brier = brier_score_loss(y_test, proba)
print(f"Доля y=1 на тесте: {y_test.mean():.3f} (при сильном дисбалансе accuracy почти бесполезна; смотрите AUC и Brier)")
print(f"Test accuracy: {acc:.3f}")
print(f"Test ROC-AUC:  {auc:.3f}")
print(f"Test Brier:    {brier:.4f}")
print(classification_report(y_test, pred, digits=3))

# Вероятность события «в течение горизонта цена закрытия превысит текущую» для последней точки каждого тикера
latest = ds.sort_values(["secid", "begin"]).groupby("secid").tail(1).copy()
latest["p_hit_within_month"] = cal_clf.predict_proba(latest[feature_cols])[:, 1]
latest[["secid", "begin", "close", "p_hit_within_month"]].sort_values(
    "p_hit_within_month", ascending=False
)

Доля y=1 на тесте: 0.834 (при сильном дисбалансе accuracy почти бесполезна; смотрите AUC и Brier)
Test accuracy: 0.834
Test ROC-AUC:  0.540
Test Brier:    0.1383
              precision    recall  f1-score   support

           0      0.000     0.000     0.000       277
           1      0.834     1.000     0.910      1393

    accuracy                          0.834      1670
   macro avg      0.417     0.500     0.455      1670
weighted avg      0.696     0.834     0.759      1670



/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

,secid,begin,close,p_hit_within_month
6655,SBER,2026-05-03,320.96,0.852178
3327,LKOH,2026-05-03,5255.00,0.843623
5822,ROSN,2026-05-03,423.20,0.843623
4160,MGNT,2026-05-03,2531.00,0.835276
7486,TATN,2026-05-03,576.70,0.835276
1665,GAZP,2026-05-03,120.66,0.806669
8315,VTBR,2026-05-03,91.03,0.794465
832,ALRS,2026-05-03,28.12,0.790459
2494,GMKN,2026-05-03,128.96,0.790459
4989,NVTK,2026-05-03,1137.60,0.790459
